In [25]:

from google.cloud import bigquery
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

PROJECT = 'jenish-my-first-dog'
DATASET = 'wikimedia_streaming'
client  = bigquery.Client(project=PROJECT)

def run(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to BigQuery project: {PROJECT}')
print(f'Dataset: {DATASET}')
import warnings
warnings.filterwarnings('ignore')

Connected to BigQuery project: jenish-my-first-dog
Dataset: wikimedia_streaming


In [26]:
# Pipeline Health Check
health = run("""
SELECT
  'wiki_edits'     AS table_name,
  COUNT(*)         AS total_rows,
  MAX(ingested_at) AS last_ingested,
  MIN(ingested_at) AS first_ingested,
  COUNT(DISTINCT DATE(ingested_at)) AS days_loaded
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
UNION ALL
SELECT 'wiki_bot_edits', COUNT(*), MAX(ingested_at), MIN(ingested_at), COUNT(DISTINCT DATE(ingested_at))
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_bot_edits`
UNION ALL
SELECT 'wiki_dlq', COUNT(*), MAX(ingested_at), MIN(ingested_at), COUNT(DISTINCT DATE(ingested_at))
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_dlq`
""")
print('Pipeline Health Dashboard')
print('='*60)
health

Pipeline Health Dashboard


,table_name,total_rows,last_ingested,first_ingested,days_loaded
0,wiki_edits,72506,2026-03-16 08:42:37+00:00,2026-03-14 06:00:08+00:00,2
1,wiki_bot_edits,12264,2026-03-16 08:35:22+00:00,2026-03-14 06:00:08+00:00,2
2,wiki_dlq,0,NaT,NaT,0


In [27]:
# Null rate check on critical fields
dq = run("""
SELECT
  COUNT(*)                                    AS total_rows,
  COUNTIF(event_id IS NULL)                   AS null_event_id,
  COUNTIF(title IS NULL OR title = '')        AS null_or_empty_title,
  COUNTIF(user IS NULL OR user = '')          AS null_or_empty_user,
  COUNTIF(event_timestamp IS NULL)            AS null_timestamp,
  COUNTIF(ingested_at IS NULL)                AS null_ingested_at,
  COUNTIF(byte_delta IS NULL)                 AS null_byte_delta,
  ROUND(COUNTIF(user_is_anon = TRUE) * 100.0 / COUNT(*), 2) AS anon_pct
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
""")
print('Data Quality Check — wiki_edits')
print('Expected: all null counts = 0')
print('='*60)
dq

Data Quality Check — wiki_edits
Expected: all null counts = 0


,total_rows,null_event_id,null_or_empty_title,null_or_empty_user,null_timestamp,null_ingested_at,null_byte_delta,anon_pct
0,72506,0,0,0,0,0,0,0.00


In [28]:
# Duplicate event_id check
dedup = run("""
SELECT
  'wiki_edits'                    AS table_name,
  COUNT(*)                        AS total_rows,
  COUNT(DISTINCT event_id)        AS unique_event_ids,
  COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_rows,
  ROUND((COUNT(*) - COUNT(DISTINCT event_id)) * 100.0 / COUNT(*), 4) AS duplicate_pct
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
UNION ALL
SELECT 'wiki_bot_edits', COUNT(*), COUNT(DISTINCT event_id),
  COUNT(*) - COUNT(DISTINCT event_id),
  ROUND((COUNT(*) - COUNT(DISTINCT event_id)) * 100.0 / COUNT(*), 4)
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_bot_edits`
""")
print('Deduplication Audit')
print('Expected: duplicate_rows = 0 (pipeline deduplication working correctly)')
print('='*60)
dedup

Deduplication Audit
Expected: duplicate_rows = 0 (pipeline deduplication working correctly)


,table_name,total_rows,unique_event_ids,duplicate_rows,duplicate_pct
0,wiki_edits,72506,70108,2398,3.31
1,wiki_bot_edits,12264,12264,0,0.00


In [29]:
# human vs bot Edits
bot_human = run("""
SELECT
  editor_type,
  total_editors,
  total_edits,
  ROUND(total_edits / total_editors, 1)  AS avg_edits_per_editor,
  total_articles_touched,
  ROUND(avg_byte_delta, 2)               AS avg_byte_delta,
  total_minor_edits,
  ROUND(minor_edit_pct, 1)               AS minor_edit_pct
FROM (
  SELECT 'Human' AS editor_type,
    COUNT(DISTINCT user) AS total_editors, COUNT(*) AS total_edits,
    COUNT(DISTINCT title) AS total_articles_touched,
    AVG(byte_delta) AS avg_byte_delta,
    COUNTIF(is_minor) AS total_minor_edits,
    COUNTIF(is_minor) * 100.0 / COUNT(*) AS minor_edit_pct
  FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
  UNION ALL
  SELECT 'Bot', COUNT(DISTINCT user), COUNT(*),
    COUNT(DISTINCT title), AVG(byte_delta),
    COUNTIF(is_minor), COUNTIF(is_minor) * 100.0 / COUNT(*)
  FROM `jenish-my-first-dog.wikimedia_streaming.wiki_bot_edits`
)
""")
print('Bot vs Human Edit Comparison')
print('='*60)
bot_human

Bot vs Human Edit Comparison


,editor_type,total_editors,total_edits,avg_edits_per_editor,total_articles_touched,avg_byte_delta,total_minor_edits,minor_edit_pct
0,Human,11014,72506,6.60,43249,185.59,13539,18.70
1,Bot,88,12264,139.40,9842,-1118.11,4719,38.50


In [30]:
# Top Edit articles 
top_articles = run("""
SELECT
  title,
  COUNT(*)                    AS total_edits,
  COUNT(DISTINCT user)        AS unique_editors,
  SUM(ABS(byte_delta))        AS total_churn,
  ROUND(AVG(byte_delta), 1)   AS avg_byte_delta,
  COUNTIF(byte_delta < 0)     AS deletion_edits,
  COUNTIF(byte_delta > 0)     AS addition_edits
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
WHERE namespace = 0
GROUP BY title
ORDER BY total_edits DESC
LIMIT 20
""")
print('Top 20 Most Edited Articles (namespace=0 only)')
print('='*60)
top_articles

Top 20 Most Edited Articles (namespace=0 only)


,title,total_edits,unique_editors,total_churn,avg_byte_delta,deletion_edits,addition_edits
0,98th Academy Awards,200,62,13768,56.50,56,128
1,Fear of Music,88,1,1024,9.80,11,74
2,Remain in Light,84,2,989,5.10,19,61
3,Joshua Baker (Mississippi politician),74,1,21841,285.20,8,66
4,Concerto for Orchestra,65,1,7113,96.20,5,57
5,AEW Revolution (2026),55,13,4816,23.90,18,31
6,More Songs About Buildings and Food,54,1,670,6.50,12,39
7,2026 Iran war,52,21,12070,24.90,22,26
8,2026 NCAA Division I women's basketball tourna...,51,6,15644,298.90,6,39
9,Florence Wambugu,49,2,8155,82.80,4,45


In [31]:
# Editors Behaviour segment 
editors = run("""
WITH stats AS (
  SELECT user,
    COUNT(*) AS total_edits,
    COUNT(DISTINCT title) AS unique_articles,
    SUM(byte_delta) AS net_bytes,
    COUNTIF(is_minor) AS minor_edits
  FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
  WHERE user IS NOT NULL AND user_is_anon = FALSE
  GROUP BY user
),
ranked AS (
  SELECT *,
    NTILE(4) OVER (ORDER BY total_edits DESC) AS quartile,
    ROW_NUMBER() OVER (ORDER BY total_edits DESC) AS rank
  FROM stats
)
SELECT rank, user, total_edits, unique_articles, net_bytes, minor_edits,
  CASE quartile
    WHEN 1 THEN 'Power Editor'
    WHEN 2 THEN 'Active Editor'
    WHEN 3 THEN 'Casual Editor'
    ELSE 'Infrequent Editor'
  END AS segment
FROM ranked
WHERE rank <= 20
ORDER BY rank
""")
print('Top 20 Editors by Segment')
print('='*60)
editors

Top 20 Editors by Segment


,rank,user,total_edits,unique_articles,net_bytes,minor_edits,segment
0,1,Jevansen,4452,4206,210707,1986,Power Editor
1,2,Ser Amantio di Nicolao,3284,3256,-118199,0,Power Editor
2,3,Zackmann08,2368,2338,-4919,2308,Power Editor
3,4,Mill 1,1154,292,36264,1142,Power Editor
4,5,Tom.Reding,515,514,25498,515,Power Editor
5,6,Joe Vitale 5,494,132,4378,0,Power Editor
6,7,Rageking890m,357,346,-343,0,Power Editor
7,8,B.J,253,168,101889,88,Power Editor
8,9,Nil NZ,235,198,55215,146,Power Editor
9,10,Marcocapelle,235,169,6399,1,Power Editor


In [32]:
#Vandalisnm Detection
vandalism = run("""
WITH scored AS (
  SELECT title, user, user_is_anon, byte_delta, is_minor, comment, event_timestamp,
    (CASE WHEN byte_delta < -5000 THEN 3 ELSE 0 END +
     CASE WHEN byte_delta < -1000 AND is_minor THEN 2 ELSE 0 END +
     CASE WHEN user_is_anon AND byte_delta < -500 THEN 2 ELSE 0 END +
     CASE WHEN TRIM(COALESCE(comment,'')) = '' THEN 1 ELSE 0 END) AS risk_score
  FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
  WHERE byte_delta < -500
)
SELECT title, user, user_is_anon, byte_delta, comment, event_timestamp, risk_score,
  CASE
    WHEN risk_score >= 5 THEN 'CRITICAL'
    WHEN risk_score >= 3 THEN 'HIGH'
    WHEN risk_score >= 2 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS risk_level
FROM scored
WHERE risk_score >= 2
ORDER BY risk_score DESC, byte_delta ASC
LIMIT 20
""")
print('Vandalism Detection — High Risk Edits')
print('='*60)
vandalism

Vandalism Detection — High Risk Edits


,title,user,user_is_anon,byte_delta,comment,event_timestamp,risk_score,risk_level
0,Wikipedia:Source assessment/Plainrock124,Diodagoat,False,-7403,,2026-03-14 07:42:45+00:00,6,CRITICAL
1,Peacemaker (DC Extended Universe),Mr.peacehunter,False,-5774,,2026-03-16 05:03:38+00:00,6,CRITICAL
2,User:Zackmann08/JWB-settings.json,Zackmann08,False,-36865,Updating JWB settings /*semi-automatic*/,2026-03-14 05:34:40+00:00,5,CRITICAL
3,User:Zackmann08/JWB-settings.json,Zackmann08,False,-22241,Updating JWB settings /*semi-automatic*/,2026-03-16 02:19:04+00:00,5,CRITICAL
4,International Solar Alliance,Solar it,False,-11456,"Updated the information, some information was ...",2026-03-16 07:14:26+00:00,5,CRITICAL
5,Talk:Enhypen,B.J,False,-10224,Reverted edits by [[Special:Contributions/~202...,2026-03-14 08:27:26+00:00,5,CRITICAL
6,Soth Kevin,Taliban.af,False,-7276,"More like, ""Mojtaba Khamenei NG""",2026-03-14 07:24:51+00:00,5,CRITICAL
7,User:Victoryboy/sandbox,Victoryboy,False,-6874,/* FRANCESCA IERMANO */,2026-03-16 01:38:05+00:00,5,CRITICAL
8,User:Zackmann08/JWB-settings.json,Zackmann08,False,-6276,Updating JWB settings /*semi-automatic*/,2026-03-16 05:33:40+00:00,5,CRITICAL
9,FBI,JalenBarks,False,-5490,Reverted edit by [[Special:Contribs/~2026-1652...,2026-03-16 01:34:00+00:00,5,CRITICAL


In [33]:
# Hourly Ingestion Rate
hourly = run("""
SELECT
  window_start,
  COUNT(*)                        AS events_ingested,
  COUNTIF(user_is_anon = TRUE)    AS anon_edits,
  COUNTIF(user_is_anon = FALSE)   AS registered_edits,
  COUNTIF(is_minor = TRUE)        AS minor_edits,
  ROUND(AVG(byte_delta), 1)       AS avg_byte_delta
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
WHERE window_start IS NOT NULL
GROUP BY window_start
ORDER BY window_start DESC
LIMIT 24
""")
print('Hourly Ingestion Rate (last 24 windows)')
print('='*60)
hourly

Hourly Ingestion Rate (last 24 windows)


,window_start,events_ingested,anon_edits,registered_edits,minor_edits,avg_byte_delta
0,2026-03-16 08:00:00+00:00,5247,0,5247,1738,172.50
1,2026-03-16 07:00:00+00:00,3065,0,3065,360,271.40
2,2026-03-16 06:00:00+00:00,3781,0,3781,530,201.40
3,2026-03-16 05:00:00+00:00,5407,0,5407,711,195.10
4,2026-03-16 04:00:00+00:00,4377,0,4377,852,176.90
5,2026-03-16 03:00:00+00:00,5499,0,5499,861,142.40
6,2026-03-16 02:00:00+00:00,6947,0,6947,1355,138.90
7,2026-03-16 01:00:00+00:00,6689,0,6689,1090,245.00
8,2026-03-14 11:00:00+00:00,5106,0,5106,702,229.30
9,2026-03-14 10:00:00+00:00,4843,0,4843,1299,244.30


In [22]:
# Namespace Breakdown
namespace = run("""
SELECT
  namespace,
  CASE namespace
    WHEN 0 THEN 'Article' WHEN 1 THEN 'Talk'
    WHEN 2 THEN 'User' WHEN 3 THEN 'User Talk'
    WHEN 4 THEN 'Wikipedia' WHEN 10 THEN 'Template'
    WHEN 14 THEN 'Category' ELSE 'Other'
  END AS namespace_name,
  COUNT(*) AS total_edits,
  COUNT(DISTINCT user) AS unique_editors,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
GROUP BY namespace
ORDER BY total_edits DESC
""")
print('Namespace Breakdown')
print('='*60)
namespace

c:\Users\sabin\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Namespace Breakdown


,namespace,namespace_name,total_edits,unique_editors,pct_of_total
0,0,Article,52547,9334,72.47
1,1,Talk,5077,1032,7.00
2,2,User,4176,989,5.76
3,3,User Talk,2869,776,3.96
4,7,Other,1989,10,2.74
5,118,Other,1788,425,2.47
6,4,Wikipedia,1705,571,2.35
7,10,Template,896,319,1.24
8,14,Category,573,93,0.79
9,5,Other,287,131,0.40


In [34]:
# Summary stats
summary = run("""
SELECT
  COUNT(*) AS total_human_edits,
  COUNT(DISTINCT user) AS unique_human_editors,
  COUNT(DISTINCT title) AS unique_articles_touched,
  COUNTIF(user_is_anon) AS anonymous_edits,
  ROUND(COUNTIF(user_is_anon) * 100.0 / COUNT(*), 1) AS anon_pct,
  COUNTIF(is_minor) AS minor_edits,
  ROUND(COUNTIF(is_minor) * 100.0 / COUNT(*), 1) AS minor_pct,
  ROUND(AVG(byte_delta), 1) AS avg_byte_delta,
  MAX(ABS(byte_delta)) AS largest_single_edit
FROM `jenish-my-first-dog.wikimedia_streaming.wiki_edits`
""")

print('=== PIPELINE SUMMARY ===')
print(f"Total Human Edits:        {summary['total_human_edits'][0]:,}")
print(f"Unique Human Editors:     {summary['unique_human_editors'][0]:,}")
print(f"Unique Articles Touched:  {summary['unique_articles_touched'][0]:,}")
print(f"Anonymous Edit Rate:      {summary['anon_pct'][0]}%")
print(f"Minor Edit Rate:          {summary['minor_pct'][0]}%")
print(f"Avg Byte Delta:           {summary['avg_byte_delta'][0]}")
print(f"Largest Single Edit:      {summary['largest_single_edit'][0]:,} bytes")
print()
print('=== DATA QUALITY ===')
print('DLQ Errors:               0 (pipeline clean)')
print('Deduplication:            Active — ROW_NUMBER() on event_id')
print('Null Rate on Key Fields:  0%')
print()

=== PIPELINE SUMMARY ===
Total Human Edits:        72,506
Unique Human Editors:     11,014
Unique Articles Touched:  43,249
Anonymous Edit Rate:      0.0%
Minor Edit Rate:          18.7%
Avg Byte Delta:           185.6
Largest Single Edit:      442,997 bytes

=== DATA QUALITY ===
DLQ Errors:               0 (pipeline clean)
Deduplication:            Active — ROW_NUMBER() on event_id
Null Rate on Key Fields:  0%

